# CISB5123 Text Analytics - Lab Assignment 3
Topic Modeling with LDA using Gensim

1. Muhammad Rayyan Farhan bin Zuhardi(SW01083761)
2. Muhammad Arif Luqman bin Mohd Zaihan(SW01083752)

1. Import the necessary libraries

In [7]:
!pip install gensim

In [8]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adamz\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adamz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adamz\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\adamz\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

2. Read the dataset

In [10]:
df = pd.read_csv('news_dataset.csv', usecols=['text'])
df.head()

,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


In [11]:
print("Dataset shape:", df.shape)
print("Number of null values in text column:", df['text'].isnull().sum())

Dataset shape: (11314, 1)
Number of null values in text column: 218


3. Text pre-processing

The steps below:
- remove null values
- convert text to lowercase
- remove punctuation and non-alphabetic characters
- tokenize text
- remove stopwords
- apply stemming
- apply lemmatization

In [12]:
df = df.dropna(subset=['text']).copy()
df['text'] = df['text'].astype(str)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and short tokens
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

df['processed_text'] = df['text'].apply(preprocess_text)
df.head()

,text,processed_text
0,I was wondering if anyone out there could enli...,"[wonder, anyon, could, enlighten, car, saw, da..."
1,I recently posted an article asking what kind ...,"[recent, post, articl, ask, kind, rate, singl,..."
2,\nIt depends on your priorities. A lot of peo...,"[depend, prioriti, lot, peopl, put, higher, pr..."
3,an excellent automatic can be found in the sub...,"[excel, automat, found, subaru, legaci, switch..."
4,: Ford and his automobile. I need information...,"[ford, automobil, need, inform, whether, ford,..."


In [13]:
# Remove empty processed rows if any
df = df[df['processed_text'].map(len) > 0].copy()

print("Number of usable documents after preprocessing:", len(df))
print("Sample processed text:")
print(df['processed_text'].iloc[0][:30]) 

Number of usable documents after preprocessing: 10996
Sample processed text:
['wonder', 'anyon', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sport', 'car', 'look', 'late', 'earli', 'call', 'bricklin', 'door', 'realli', 'small', 'addit', 'front', 'bumper', 'separ', 'rest', 'bodi', 'know', 'anyon', 'tellm', 'model', 'name', 'engin']


4. Create dictionary and corpus for LDA

In [15]:
dictionary = corpora.Dictionary(df['processed_text'])

# Filter out very rare and very common words
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in df['processed_text']]

print("Number of unique tokens:", len(dictionary))
print("Number of documents:", len(corpus))

Number of unique tokens: 10984
Number of documents: 10996


5. Perform LDA using Gensim (4topics)

In [16]:
num_topics = 4

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# Display the discovered topics
for idx, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {idx + 1}: {topic}")
    print()

Topic 1: 0.018*"use" + 0.016*"key" + 0.012*"max" + 0.010*"encrypt" + 0.009*"file" + 0.008*"system" + 0.007*"chip" + 0.006*"program" + 0.006*"bit" + 0.006*"secur"

Topic 2: 0.011*"edu" + 0.009*"team" + 0.009*"game" + 0.007*"year" + 0.006*"com" + 0.006*"play" + 0.005*"new" + 0.005*"univers" + 0.004*"season" + 0.004*"leagu"

Topic 3: 0.010*"peopl" + 0.008*"would" + 0.007*"one" + 0.006*"govern" + 0.006*"say" + 0.005*"law" + 0.005*"think" + 0.005*"god" + 0.004*"right" + 0.004*"state"

Topic 4: 0.011*"get" + 0.011*"one" + 0.010*"would" + 0.009*"like" + 0.008*"know" + 0.007*"think" + 0.007*"time" + 0.006*"use" + 0.006*"good" + 0.006*"work"



6. Evaluate the LDA model using Coherence Score

In [17]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=df['processed_text'],
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", round(coherence_score, 4))

Coherence Score: 0.5431


7. Assign dominant topic to each document

In [19]:
def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow)
    return max(topic_probs, key=lambda x: x[1])[0]

df['dominant_topic'] = [get_dominant_topic(bow) for bow in corpus]

df[['text', 'dominant_topic']].head(10)

,text,dominant_topic
0,I was wondering if anyone out there could enli...,3
1,I recently posted an article asking what kind ...,3
2,\nIt depends on your priorities. A lot of peo...,3
3,an excellent automatic can be found in the sub...,3
4,: Ford and his automobile. I need information...,3
5,\nYo! Watch the attributions--I didn't say tha...,2
6,\nYou can avoid these problems entirely by ins...,3
7,"I have a 1986 Acura Integra 5 speed with 95,00...",3
8,"\nassuming yours is a non turbo MR2, the gruff...",3
9,"\n\nIn addition to restricted mileage, many cl...",3


8. Interpretation of the coherence score


The coherence score is used to measure how semantically meaningful the topics are based on the words grouped within each topic. In general, a higher coherence score indicates that the generated topics are more interpretable and the top words in each topic are more related to one another. In this assignment, the coherence score obtained from the LDA model suggests that the model produces a reasonably meaningful topic structure for the news dataset. However, the score may still be improved by adjusting the preprocessing steps, tuning the number of topics, or changing the LDA parameters such as passes and alpha.


9. Short conclusion

This notebook successfully performs topic modeling on the `news_dataset.csv` dataset using LDA with Gensim. The text data is preprocessed through stopword removal, stemming, and lemmatization before training the model. Then, the coherence score is calculated to evaluate the quality of the discovered topics. Finally, the topics can be interpreted based on their top keywords and dominant topic assignments.
